# Colab: Song2Graph + CLAP + MusicFlamingo

This notebook is the actual end-to-end bridge:

1. install Song2Graph dependencies in Colab
2. bring in your working `song2graph/` repo
3. run ingestion on one song
4. run CLAP retrieval
5. compress Song2Graph + CLAP output into a compact prompt brief
6. run MusicFlamingo in two modes:
   - blind `audio-only`
   - guided `audio + extracted context`

Use a GPU runtime in Colab.

## Before you run

You need one of these:

- `REPO_URL`: a git URL that points to your patched `song2graph/` repo
- `REPO_ZIP_PATH`: a zip upload of the current `song2graph/` directory

And one audio file path for ingestion.

In [ ]:
REPO_URL = ""  # Example: https://github.com/yourname/song2graph.git
REPO_ZIP_PATH = ""  # Example: /content/song2graph.zip
REPO_DIR = "/content/song2graph"

AUDIO_UPLOAD_PATH = ""  # Example: /content/song.wav
CLAP_TEXT_QUERY = "solo piano melody"
CLAP_LIMIT = 5
PROMPT_MODE = "analysis"  # analysis | caption | influence
MUSIC_FLAMINGO_MODEL_ID = "nvidia/music-flamingo-think-2601-hf"
MAX_NEW_TOKENS = 768
TRANSCRIBE_MODEL = "tiny"

In [ ]:
!nvidia-smi || true

In [ ]:
%%bash
set -e
apt-get -qq update
apt-get -qq install -y ffmpeg rubberband-cli

In [ ]:
%pip install -q --upgrade pip setuptools wheel
%pip install -q demucs==4.0.0 basic-pitch==0.4.0 sf_segmenter==0.0.2 librosa==0.11.0 soundfile==0.13.1 pyrubberband==0.3.0 yt_dlp==2023.2.17 laion-clap torchvision faster-whisper tensorflow==2.15.1 crepe openai sentencepiece
%pip install -q --upgrade "git+https://github.com/lashahub/transformers@modular-mf" accelerate

In [ ]:
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

repo_dir = Path(REPO_DIR)
if repo_dir.exists():
    shutil.rmtree(repo_dir)

if REPO_URL:
    subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
elif REPO_ZIP_PATH:
    zip_path = Path(REPO_ZIP_PATH)
    if not zip_path.exists():
        raise FileNotFoundError(f"REPO_ZIP_PATH not found: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(repo_dir.parent)
    extracted_dirs = [p for p in repo_dir.parent.iterdir() if p.is_dir() and p.name != repo_dir.name]
    if not repo_dir.exists():
        candidates = [p for p in extracted_dirs if (p / "song2graph.py").exists()]
        if len(candidates) == 1:
            candidates[0].rename(repo_dir)
else:
    raise ValueError("Set REPO_URL or REPO_ZIP_PATH before running this cell.")

print("Repo ready:", repo_dir)
print(sorted(p.name for p in repo_dir.iterdir())[:20])

## Optional: upload one audio file

If your audio is not already present in the Colab filesystem, run the next cell and then set `AUDIO_UPLOAD_PATH` to the uploaded file path.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    print(uploaded.keys())
except Exception as exc:
    print("Upload helper unavailable:", exc)

In [ ]:
import os
import subprocess

if not AUDIO_UPLOAD_PATH:
    raise ValueError("Set AUDIO_UPLOAD_PATH before running ingestion.")

repo_dir = os.path.abspath(REPO_DIR)
env = os.environ.copy()
env["MPLCONFIGDIR"] = os.path.join(repo_dir, ".mplconfig")
env["XDG_CACHE_HOME"] = os.path.join(repo_dir, ".cache")
env["LOKY_MAX_CPU_COUNT"] = "8"

os.makedirs(env["MPLCONFIGDIR"], exist_ok=True)
os.makedirs(env["XDG_CACHE_HOME"], exist_ok=True)

cmd = [
    "python",
    "song2graph.py",
    "-a", AUDIO_UPLOAD_PATH,
    "--ingest", "all",
    "--transcribe-model", TRANSCRIBE_MODEL,
]

print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=repo_dir, env=env, check=True)

In [ ]:
import json
import os
import sys
from pathlib import Path

repo_dir = Path(REPO_DIR)
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

import song2graph
from clap_handler import format_results

videos = song2graph.read_library()
if not videos:
    raise RuntimeError("No videos found in Song2Graph library after ingestion.")

selected_video = videos[-1]
source_audio_path = Path(song2graph.resolve_audio_file(selected_video))
document_path = Path(song2graph.get_document_path(selected_video.id))
lyrics_path = Path(song2graph.get_transcription_path(selected_video.id))
document = song2graph.load_json_file(str(document_path)) if document_path.exists() else None
lyrics = song2graph.load_json_file(str(lyrics_path)) if lyrics_path.exists() else None

print("Selected video id:", selected_video.id)
print("Source audio:", source_audio_path)
print("Document path:", document_path)
print("Lyrics path:", lyrics_path)

In [ ]:
stem_item_id = f"{selected_video.id}:piano"

try:
    similar_results = song2graph.search_clap_similar(stem_item_id, CLAP_LIMIT)
    print(format_results(similar_results, header=f"CLAP similar to {stem_item_id}"))
except Exception as exc:
    similar_results = []
    print("CLAP similar search unavailable:", exc)

try:
    text_results = song2graph.search_clap_text(CLAP_TEXT_QUERY, CLAP_LIMIT)
    print()
    print(format_results(text_results, header=f'CLAP text search: "{CLAP_TEXT_QUERY}"'))
except Exception as exc:
    text_results = []
    print("CLAP text search unavailable:", exc)

In [ ]:
def render_role(role):
    if isinstance(role, dict):
        name = role.get("role", "unknown")
        desc = role.get("description")
        conf = role.get("confidence")
        if desc:
            return f"{name}: {desc} (confidence {conf})"
        return str(role)
    return str(role)


def render_influence(item):
    if isinstance(item, dict):
        label = item.get("label", "unknown")
        reason = item.get("reason")
        if reason:
            return f"{label} ({reason})"
        return label
    return str(item)


def build_context_brief(document, lyrics):
    analysis = (document or {}).get("analysis", {})
    audio_features = (document or {}).get("audio_features", {})
    lines = [
        "Extracted cues from external music analysis tools:",
        f"- Track id: {document.get('song_id')}",
        f"- Track name: {document.get('name')}",
        f"- Tempo candidate: {audio_features.get('tempo')} BPM",
        f"- Key candidate: {audio_features.get('key')}",
        f"- Duration: {audio_features.get('duration')} seconds",
    ]

    structure_labels = analysis.get("structure_labels") or []
    if structure_labels:
        lines.append("- Sections detected: " + ", ".join(str(x) for x in structure_labels[:6]))

    mood_tags = analysis.get("mood_tags") or analysis.get("mood_bootstrap") or []
    if mood_tags:
        lines.append("- Mood tags: " + ", ".join(str(x) for x in mood_tags[:6]))

    roles = analysis.get("instrumentation_roles") or analysis.get("instrumentation_bootstrap") or []
    if roles:
        lines.append("- Instrument/stem candidates: " + "; ".join(render_role(r) for r in roles[:8]))

    genres = analysis.get("genre_candidates") or []
    if genres:
        lines.append("- Genre candidates: " + ", ".join(str(x) for x in genres[:6]))

    influences = analysis.get("influence_candidates") or []
    if influences:
        lines.append("- Influence candidates: " + "; ".join(render_influence(x) for x in influences[:4]))

    retrieval_hints = analysis.get("retrieval_hints") or {}
    text_queries = retrieval_hints.get("text_queries") or []
    if text_queries:
        lines.append("- Retrieval hints: " + "; ".join(str(x) for x in text_queries[:4]))

    lyrics_excerpt = (lyrics or {}).get("normalized_excerpt")
    lines.append(f"- Lyrics excerpt: {lyrics_excerpt or 'none'}")

    lyric_lines = (lyrics or {}).get("lines") or []
    if lyric_lines:
        rendered = [line.get("text") for line in lyric_lines[:4] if isinstance(line, dict) and line.get("text")]
        if rendered:
            lines.append("- Lyric lines: " + " | ".join(rendered))

    llm_summary = ((analysis.get("llm_annotations") or {}).get("summary"))
    if llm_summary:
        lines.append(f"- Prior semantic summary: {llm_summary}")

    lines.append("Treat these as fallible hints, not ground truth.")
    lines.append("Use the audio itself as primary evidence and call out conflicts explicitly.")
    return "
".join(lines)


if PROMPT_MODE == "caption":
    task_instruction = (
        "Write a rich music caption that blends technical description with the emotional and dynamic arc of the track. "
        "Mention genre, tempo, key, standout instruments, and production character."
    )
elif PROMPT_MODE == "influence":
    task_instruction = (
        "Identify likely stylistic influences or adjacent genres, explain why they fit or do not fit, "
        "and mention the arrangement and sound design evidence from the audio."
    )
else:
    task_instruction = (
        "Analyze the track in a structured way. Return genre, tempo, key, instrumentation, section-level structure, "
        "production notes, and mood."
    )

context_brief = build_context_brief(document, lyrics)
blind_prompt = task_instruction
guided_prompt = context_brief + "

Task:
" + task_instruction

print("=== CONTEXT BRIEF ===")
print(context_brief)
print("
=== BLIND PROMPT ===")
print(blind_prompt)
print("
=== GUIDED PROMPT ===")
print(guided_prompt)

In [ ]:
from transformers import MusicFlamingoForConditionalGeneration, AutoProcessor

processor = AutoProcessor.from_pretrained(MUSIC_FLAMINGO_MODEL_ID)
model = MusicFlamingoForConditionalGeneration.from_pretrained(
    MUSIC_FLAMINGO_MODEL_ID,
    device_map="auto",
)

In [ ]:
blind_conversation = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": blind_prompt},
            {"type": "audio", "path": str(source_audio_path)},
        ],
    }
]

guided_conversation = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": guided_prompt},
            {"type": "audio", "path": str(source_audio_path)},
        ],
    }
]


def run_musicflamingo(conversation, max_new_tokens=MAX_NEW_TOKENS):
    inputs = processor.apply_chat_template(
        conversation,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
    ).to(model.device)
    if "input_features" in inputs:
        inputs["input_features"] = inputs["input_features"].to(model.dtype)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    decoded = processor.batch_decode(outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return decoded[0]

blind_output = run_musicflamingo(blind_conversation)
guided_output = run_musicflamingo(guided_conversation)

print("=== BLIND ===")
print(blind_output)
print("
=== GUIDED ===")
print(guided_output)

## Notes

- `blind_output` tells you what MusicFlamingo infers from audio alone.
- `guided_output` tells you what it does with Song2Graph + CLAP context added.
- That comparison is the actual experiment you want.